# Phase 10: Leakage-Safe Preprocessing Pipeline Architecture

**Study**: *Student Performance Prediction Using Machine Learning and Explainable AI: An Experimental Study Using the Open University Learning Analytics Dataset (OULAD)*

### Methodological Requirements for Phase 10:
1. **Strict Training-Fold Isolation**: All imputation medians, one-hot encoding categories, and scaling parameters are fitted **strictly and exclusively** on the training partition ($N = 22,815$).
2. **Zero Test-Data Peek**: Test data ($N = 4,889$) and Validation data ($N = 4,889$) are transformed purely via forward projection using learned training statistics.
3. **Model-Specific Transformers**:
   - **Linear / Distance Models**: `SimpleImputer(median)` $\rightarrow$ `StandardScaler` $\rightarrow$ `OneHotEncoder`.
   - **Tree-Based Models**: `SimpleImputer(median)` $\rightarrow$ `OneHotEncoder` (`StandardScaler` intentionally omitted to preserve natural split boundaries).
4. **Score Representation**: Unsubmitted early assessments retain `NaN` representation alongside explicit submission indicator `has_early_assessment_submission_d30`.
5. **Registration Date Restoration**: Any legacy global median in `date_registration` is reverted to `NaN` prior to pipeline entry, ensuring fold-level median calculation.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
SPLITS_DIR = os.path.join(PROJECT_ROOT, "results", "splits")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from preprocessing import run_preprocessing_pipeline, build_column_transformer, get_scenario_feature_lists

specs, audit_results = run_preprocessing_pipeline(PROCESSED_DIR, SPLITS_DIR, TABLES_DIR)
display(pd.DataFrame(specs))
display(pd.DataFrame(audit_results))

In [ ]:
# Programmatic Inspection of Learned Parameters on Scenario S0
sc_s0_path = os.path.join(PROCESSED_DIR, "scenario_s0.csv")
df_s0 = pd.read_csv(sc_s0_path)

# Restore true raw missingness for date_registration
if 'date_registration_imputed_flag' in df_s0.columns:
    df_s0.loc[df_s0['date_registration_imputed_flag'] == 1, 'date_registration'] = np.nan

num_cols, cat_cols = get_scenario_feature_lists('S0')
features = num_cols + cat_cols

X_train = df_s0[df_s0['partition'] == 'train'][features].copy()
X_val = df_s0[df_s0['partition'] == 'validation'][features].copy()
X_test = df_s0[df_s0['partition'] == 'test'][features].copy()

linear_ct = build_column_transformer('S0', model_family='linear')
linear_ct.fit(X_train)

imputer = linear_ct.named_transformers_['num'].named_steps['imputer']
scaler = linear_ct.named_transformers_['num'].named_steps['scaler']
encoder = linear_ct.named_transformers_['cat'].named_steps['onehot']

print("=== S0 LEARNED PARAMETERS (TRAINING PARTITION ONLY) ===")
for col, med, mean, std in zip(num_cols, imputer.statistics_, scaler.mean_, scaler.scale_):
    print(f"  Numerical '{col}': Median = {med:.2f}, Mean = {mean:.2f}, Std = {std:.2f}")

print(f"\nOneHotEncoder Categories learned across {len(cat_cols)} categorical predictors:")
for col, cats in zip(cat_cols, encoder.categories_):
    print(f"  '{col}': {len(cats)} categories -> {list(cats)}")

X_train_trans = linear_ct.transform(X_train)
X_val_trans = linear_ct.transform(X_val)
X_test_trans = linear_ct.transform(X_test)

print(f"\nTransformed Shape Train: {X_train_trans.shape}")
print(f"Transformed Shape Val:   {X_val_trans.shape}")
print(f"Transformed Shape Test:  {X_test_trans.shape}")
assert X_train_trans.shape[1] == 52
print("\nS0 PREPROCESSING VERIFICATION: ALL ASSERTIONS PASSED!")